# Laya V4.5 — Minimal attention architecture, hardware-aware pruning

Based on V4.3 and the T4 V4.4 window benchmark. **No change to pretrained FFNs, residuals, norms, RoPE or Q/K/V.** This experiment removes CeNN correction, landmarks, global memory, dynamic routing, and unnecessary FULL replacements. It only accepts a SLIDING candidate if whole-model PROBE fidelity and repeated GPU latency both pass. **Unchanged native model remains the default deployment reference.** No claimed acceleration without a final AUDIT speed pass.

This is a *pruning and selection ablation*, not a novel faster kernel. Longer context, memory use, and generation need separate validation.

In [ ]:
#@title 1. Install pinned V4.3 / V3.1-compatible dependencies and GPU check
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU (T4 or better)."
print("CUDA",torch.version.cuda,"GPU",torch.cuda.get_device_name(0),"Torch",torch.__version__)


In [ ]:
#@title 2. Fidelity-first V3.1 attention replacement configuration
from pathlib import Path
from contextlib import contextmanager
import math,copy,json,random,hashlib,time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES,build_checkpoint_model,build_sequence,collate_items,
    normalize_question,full_attention_indices,
)
from tinycenn_lm.decision_training import require_finite
from tinycenn_lm.laya_lab.pdelta import PDelta3GDN2CLVRAttention
from tinycenn_lm.laya_lab.core import _depthwise_sequence_conv,_valid_tokens
from tinycenn_lm.sliding_window_fusion import SlidingWindowFusion
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb

SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
SEED=2043
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=500 #@param {type:"integer"}
DEV_CASES=240 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
RANK=96 #@param {type:"integer"}
STEPS_FULL=10 #@param {type:"integer"}
STEPS_SLIDING=10 #@param {type:"integer"}
LOCAL_TRAIN_BATCHES=16 #@param {type:"integer"}
LOCAL_PROBE_BATCHES=4 #@param {type:"integer"}
PROBE_DECISIONS=120 #@param {type:"integer"}
AUDIT_DECISIONS=200 #@param {type:"integer"}
LEARNING_RATE=0.001 #@param {type:"number"}
OUTPUT_LR=0.00001 #@param {type:"number"}
RUN_JOINT_RECOVERY=False #@param {type:"boolean"}
JOINT_STEPS=80 #@param {type:"integer"}
JOINT_LR=0.00002 #@param {type:"number"}
JOINT_CHECK_EVERY=20 #@param {type:"integer"}
FULL_WINDOW=1024 #@param {type:"integer"}
SLIDING_RADIUS=64 #@param {type:"integer"}
SLIDING_BLOCK_SIZE=64 #@param {type:"integer"}
LANDMARK_STRIDE=16
MAX_LANDMARKS=32
FEATURE_DIM=RANK
TARGET_COSINE=.99
TARGET_NMSE=.025
MAX_LOCAL_NMSE=.025
LOG_EVERY=25 #@param {type:"integer"}
BENCH_REPEATS=8 #@param {type:"integer"}
SELECTION_MODE="quality_gated" #@param ["quality_gated", "research_all"]
MIN_AGREEMENT=.99 #@param {type:"number"}
MAX_GOLD_LOSS_PP=1.5 #@param {type:"number"}
REQUIRE_SPEEDUP_IN_GATED_MODE=True #@param {type:"boolean"}
RESUME=False #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE=False #@param {type:"boolean"}

OUT=Path("/content/Laya_V45_Lean_Attention")
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT=Path("/content/drive/MyDrive/TinyCeNN-LM/V45")
OUT.mkdir(parents=True,exist_ok=True)
assert 1<=BATCH<=4 and 16<=RANK<=256
assert 10<=STEPS_FULL and 10<=STEPS_SLIDING and LOCAL_TRAIN_BATCHES>=2
assert LOG_EVERY>=1 and BENCH_REPEATS>=2 and TRAIN_CASES+DEV_CASES<=1200
assert SELECTION_MODE in ("research_all","quality_gated")
assert FULL_WINDOW>=2*MAX_LEN-1,("V4.3 full-window teacher-parity control requires unrestricted attention",FULL_WINDOW,MAX_LEN)
assert SLIDING_RADIUS==64, "V4.3 parity control must use native radius 64"
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
        "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"
    ]
))
teacher,source_cfg=build_checkpoint_model(source_dir)
teacher.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
ORIGINAL_LAYERS=tuple(teacher.encoder.layers)
ORIGINAL_ATTN=tuple(layer.attn for layer in ORIGINAL_LAYERS)
FULL=tuple(full_attention_indices(teacher))
SLIDING=tuple(i for i in range(28) if i not in FULL)
assert FULL==(0,3,6,9,12,15,18,21,24,27)
assert len(SLIDING)==18
assert int(teacher.encoder.config.local_attention)//2==64
assert MAX_LEN<=int(source_cfg.get("max_len",MAX_LEN))
ORDER=FULL+SLIDING
print("PHASE A full:",FULL)
print("PHASE B sliding:",SLIDING)
print("One original Laya model frozen. T4 GPU",torch.cuda.get_device_name(0))
print("Selection mode:",SELECTION_MODE,"experimental forced-all" if SELECTION_MODE=="research_all" else "gated")


In [ ]:
#@title 3. Case-disjoint TRAIN / PROBE / AUDIT data + batch utilities
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V4.2 teacher helper")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items)>=max(PROBE_DECISIONS,LOCAL_PROBE_BATCHES*BATCH),"Insufficient probe decisions"
assert len(audit_items)>=AUDIT_DECISIONS,"Insufficient audit decisions"
print("official test remains untouched")


In [ ]:
#@title 4. Lean candidate architecture: no CeNN branches or FULL replacement
# Native FULL attention has already been shown to be faster than the
# V4.3 landmark/global-memory/routing stack. Preserve all FULL layers exactly.
# SlidingWindowFusion remains a *candidate* rather than an automatic deployment.
# This architecture has no learned auxiliary branch, no extra correction MLP,
# no landmark pooling, no GDN, no depthwise convolution and no joint recovery.
@contextmanager
def install_plan(plan):
    assert all(teacher.encoder.layers[i].attn is ORIGINAL_ATTN[i] for i in range(28))
    try:
        for i, adapter in plan.items():
            assert i in SLIDING, 'FULL native attention must be retained'
            teacher.encoder.layers[i].attn=adapter
        yield teacher
    finally:
        for i in range(28):
            teacher.encoder.layers[i].attn=ORIGINAL_ATTN[i]

def new_adapter(i):
    assert i in SLIDING
    return SlidingWindowFusion(ORIGINAL_ATTN[i],radius=SLIDING_RADIUS,
                               block_size=SLIDING_BLOCK_SIZE).to(device).eval()

print('FULL layers preserved:', FULL)
print('SLIDING candidates:', SLIDING)


In [ ]:
#@title 6. Whole-model PROBE fidelity + interleaved CUDA speed, without AUDIT leakage
@torch.inference_mode()
def get_reference(items,limit):
    output=[]
    for start in range(0,min(limit,len(items)),BATCH):
        selected=items[start:min(start+BATCH,limit)]
        b=direct_batch(selected)
        with torch.autocast("cuda",dtype=amp_dtype):
            scores,_=teacher(**b)
        probs=torch.softmax(scores.float().masked_fill(
            ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()
        for j,item in enumerate(selected):
            output.append(dict(probs=probs[j],teacher=int(probs[j].argmax()),
                               gold=int(item["gold_index"]),
                               case=item["case_id"],qid=item["qid"]))
    return output

@torch.inference_mode()
def evaluate(plan,items,reference):
    n=len(reference)
    agree=correct=tcorrect=0;js=0.
    with install_plan(plan):
        for start in range(0,n,BATCH):
            selected=items[start:min(n,start+BATCH)]
            b=direct_batch(selected)
            with torch.autocast("cuda",dtype=amp_dtype):
                score,_=teacher(**b)
            p=torch.softmax(score.float().masked_fill(
                ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()
            for j,item in enumerate(selected):
                t=reference[start+j]["probs"];s=p[j]
                assert s.shape==t.shape
                m=(s+t)/2
                js+=float(.5*((s*(s.clamp_min(1e-8).log()-
                                    m.clamp_min(1e-8).log())).sum()+
                               (t*(t.clamp_min(1e-8).log()-
                                    m.clamp_min(1e-8).log())).sum()))
                guess=int(s.argmax())
                agree+=int(guess==reference[start+j]["teacher"])
                correct+=int(guess==item["gold_index"])
                tcorrect+=int(reference[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,
                gold_accuracy=correct/n,teacher_gold_accuracy=tcorrect/n,
                gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(before,after,repeats=BENCH_REPEATS):
    """Interleaved exact-shape A/B/B/A, median per shape, never invent speed.

    V4.2 aggregated noisy individual ratios: the median ratio could
    contradict its displayed median native/candidate latency. This
    reports paired native/converted MEDIANS for each input shape and
    a conservative minimum speedup across all shapes.
    """
    per_shape=[]
    for batch_index,b in enumerate(BENCH_BATCHES):
        def once(plan):
            with install_plan(plan),torch.autocast("cuda",dtype=amp_dtype):
                a=torch.cuda.Event(enable_timing=True)
                z=torch.cuda.Event(enable_timing=True)
                a.record();teacher(**b);z.record();z.synchronize()
                return float(a.elapsed_time(z))
        for _ in range(5):
            once(before);once(after)
        readings={"before":[],"after":[]}
        for step in range(repeats):
            order=("before","after","after","before") if step%2==0 else (
                "after","before","before","after")
            for label in order:
                readings[label].append(once(
                    before if label=="before" else after))
        old=float(np.median(readings["before"]))
        new=float(np.median(readings["after"]))
        per_shape.append(dict(
            shape_index=batch_index,
            batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            before_ms=old,
            after_ms=new,
            speedup=old/max(new,1e-9),
            before_p95_ms=float(np.percentile(readings["before"],95)),
            after_p95_ms=float(np.percentile(readings["after"],95)),
            n=len(readings["before"])))
    speedups=[r["speedup"] for r in per_shape]
    return dict(
        median_speedup=float(np.median(speedups)),
        min_shape_speedup=float(min(speedups)),
        all_shapes_faster=bool(all(speed>1 for speed in speedups)),
        before_ms=float(np.median([r["before_ms"] for r in per_shape])),
        after_ms=float(np.median([r["after_ms"] for r in per_shape])),
        per_shape=per_shape)

PROBE_REF=get_reference(probe_items,PROBE_DECISIONS)
native_probe=evaluate({},probe_items,PROBE_REF)
assert native_probe["teacher_agreement"]==1 and native_probe["mean_js"]<1e-7
print("Original probe",native_probe)
print("Benchmark shapes",[(b["input_ids"].shape[0],b["input_ids"].shape[1])
                           for b in BENCH_BATCHES])


In [ ]:
#@title 6. Strict hardware-aware greedy search (PROBE only)
# This is intentionally inference-only: V4.3 trained zero adapter steps in all
# 28 layers and joint recovery selected the unmodified checkpoint.
# Actual speed, not cosine fidelity alone, determines whether a candidate stays.
MIN_SPEEDUP=1.05 #@param {type:'number'}
MIN_LOCAL_SPEEDUP=1.02 #@param {type:'number'}
MIN_SPEEDUP=max(float(MIN_SPEEDUP),1.0)
PLAN={}
records=[]
base_pair=benchmark_paired({}, {}, repeats=BENCH_REPEATS)
print('Native-vs-native timing control:',base_pair['median_speedup'])
for i in SLIDING:
    candidate=new_adapter(i)
    trial={**PLAN,i:candidate}
    # Full-model speed gate first: save inference cost for demonstrably slower variants.
    paired=benchmark_paired(PLAN,trial,repeats=BENCH_REPEATS)
    speed_ok=(paired['all_shapes_faster'] and
              paired['min_shape_speedup']>=MIN_LOCAL_SPEEDUP and
              paired['median_speedup']>=MIN_SPEEDUP)
    if speed_ok:
        quality=evaluate(trial,probe_items,PROBE_REF)
        ok=(quality['teacher_agreement']>=MIN_AGREEMENT and
            quality['gold_drop']<=MAX_GOLD_LOSS_PP/100 and
            quality['mean_js']<=0.003)
    else:
        quality=None
        ok=False
    if ok:
        PLAN[i]=candidate
    rec=dict(layer=i,accepted=ok,speed_gate=speed_ok,
             paired_cuda=paired,probe=quality)
    records.append(rec)
    print(f'L{i:02d}: speed {paired["median_speedup"]:.3f}x, '
          f'min {paired["min_shape_speedup"]:.3f}x; accepted={ok}')
    (OUT/'v45_selection_progress.json').write_text(json.dumps(
        dict(selected=list(PLAN),records=records),indent=2))
    if not ok:
        del candidate
print('Lean architecture selected:',list(PLAN))


In [ ]:
#@title 7. Unseen AUDIT once (never select candidates using AUDIT)
# Probe-only selection is frozen at this point.
audit_reference=get_reference(audit_items,AUDIT_DECISIONS)
assert {x['case'] for x in audit_reference}.isdisjoint(
    {x['case'] for x in PROBE_REF})
final_quality=evaluate(PLAN,audit_items,audit_reference)
final_speed=benchmark_paired({},PLAN,repeats=max(12,BENCH_REPEATS))
final_gates=dict(
    fidelity=final_quality['teacher_agreement']>=MIN_AGREEMENT,
    gold=final_quality['gold_drop']<=MAX_GOLD_LOSS_PP/100,
    speed=bool(PLAN) and final_speed['all_shapes_faster'] and
          final_speed['min_shape_speedup']>=MIN_LOCAL_SPEEDUP and
          final_speed['median_speedup']>=MIN_SPEEDUP,
    real_replacement=bool(PLAN))
final_gates['all_pass']=all(final_gates.values())
report=dict(version='V4.5-lean',source='V4.3',selected_layers=list(PLAN),
            architecture='native full + speed-gated native sliding replacements',
            removed=['CeNN correction','landmarks','global linear memory',
                     'dynamic routing','depthwise convolution','joint recovery',
                     'full layer adapters'],
            selection=records,audit=final_quality,paired_cuda=final_speed,
            gates=final_gates,
            gpu=torch.cuda.get_device_name(0))
(OUT/'v45_lean_report.json').write_text(json.dumps(report,indent=2))
print('AUDIT',final_quality)
print('FULL MODEL SPEED',final_speed['median_speedup'],'x')
print('GATES',final_gates)
if not final_gates['all_pass']:
    print('No validated faster student. Keep original native checkpoint.')
